# Introduction

* Agents can be used to perform some specific Tasks.
*  Agents are systems that use LLMs as reasoning engines to determine which actions to take and the inputs to pass them. 

* **LLM agents** using LangChain are components that leverage Large Language Models (LLMs) to perform complex tasks through natural language processing. Here’s a brief overview:

* **Purpose**: LLM agents are designed to ``automate decision-making`` and ``perform tasks`` by interpreting user input and generating responses or actions based on that input.

**Components**:

* LLMs: The backbone, providing language understanding and generation capabilities.
* Chains: Sequences of actions or tasks that the agent can perform, often structured to handle different types of inputs and outputs.
* Tools: External APIs or functionalities that the agent can call upon, enhancing its capabilities.

* Use Cases: LLM agents can be used for chatbots, question answering systems, data analysis, content generation, and more, making them versatile in various applications.

In summary, LLM agents in LangChain provide a framework for creating intelligent systems that understand and respond to human language, automating complex workflows and tasks.

# Installation

In [68]:
# %pip install -U langchain-community langgraph langchain-anthropic tavily-python langgraph-checkpoint-sqlite

# Define Tools

We will be using **Tavily (a search engine)** as a tool.

Tavily is a specialized search engine designed to optimize information retrieval for Large Language Models (LLMs) and AI agents. Its primary offering is the Tavily Search API, which enables developers to perform efficient, real-time searches that return highly relevant results tailored for AI applications.

In [6]:
import os

os.environ['TAVILY_API_KEY'] = ' '

*  built-in tool in LangChain to easily use Tavily search engine as tool.

In [11]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_result=5)

result = search.invoke('SENSEX CLOSING BENCHMARK')

print(result,'\n\n')


# If we want, we can create other tools.
# Once we have all the tools we want, we can put them in a list that we will reference later.
tools = [search]

[{'url': 'https://m.economictimes.com/markets/stocks/news/sensex-soars-900-points-nifty-tops-22750-8-key-drivers-fueling-this-rally/articleshow/119147684.cms', 'content': 'The benchmark BSE Sensex added 1131.31 points or 1.53% to close at 75,301.26, while the broader Nifty 50 index closed at 22,834.30, higher by'}, {'url': 'https://www.moneycontrol.com/news/business/markets/stock-market-live-sensex-nifty-50-share-price-gift-nifty-latest-updates-18-03-2025-liveblog-12967254.html', 'content': 'At close, the Sensex was up 1,131.31 points or 1.53 percent at 75,301.26, and the Nifty was up 325.55 points or 1.45 percent at 22,834.30. We'}, {'url': 'https://www.livemint.com/market/stock-market-news/sensex-jumps-750-points-nifty-50-reclaims-22-700-why-is-indian-stock-market-rising-explained-with-5-key-factors-11742272555154.html', 'content': "India's stock market benchmarks, the Sensex and Nifty 50, experienced significant gains on March 18. The Sensex closed 1,131 points, or 1.53"}, {'url': '

In [13]:
len(result)

5

In [15]:
result[3]

{'url': 'https://economictimes.indiatimes.com/markets/indices/bse-sensex',
 'content': "What does BSE Sensex represent?\nSensex is India's benchmark stock index which tracks 30 leading companies listed on the Bombay Stock Exchange (BSE), reflecting overall market sentiment and economic health, crucial for gauging India's stock market performance..\n\n\nWhat was the previous day's closing value of BSE Sensex?\nThe previous day's closing value of BSE Sensex was 76171.08. As of today, the current value of BSE Sensex stands at 76138.97. [...] Switch To ET\nStock\nLearn Value InvestingSubscribe\nBSE Sensex\nView all Indices\n76,138.97\n-32.11\xa0(-0.04%)\nAs on 13 Feb, 2025 16:10 IST\nSensex is India's benchmark stock index which tracks 30 leading companies listed on the Bombay Stock Exchange (BSE), reflecting overall market sentiment and economic health, crucial for gauging India's stock market performance.\n\n1D\n1W\n1M\n3M\n6M\n1Y\n3Y\n5Y [...] Key Metrics\nOpen\n76,201.1\nDay High\n76,7

In [17]:
tools

[TavilySearchResults(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********')))]

### Using Large language model to call tool

let's learn how to use a language model by to call tools. LangChain supports many different language models that you can use interchangably - select the one you want to use below!

In [1]:
# os.environ["OPENAI_API_KEY"]='YOUR_OPENAI_API_KEY'

In [70]:
# from langchain_openai import ChatOpenAI

# model = ChatOpenAI(model="gpt-4")

#### Using mistral model from togther AI

In [19]:

import getpass
import os

# if not os.environ.get("596ae3ea76ebfcfeb143a4a503b06049a168023dcc8dc44d16b2fd4653f1670b"):
#   os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter API key for Together AI: ")

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=os.environ["TOGETHERAI_API_KEY"],
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
)

In [21]:
#simple model calling
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content="hi! John here")])
response.content

"Hello John! It's nice to meet you. Is there something I can help you with or would you like to chat?"

We can now see what it is like to enable this model to do tool calling. In order to enable that we use .bind_tools to give the language model knowledge of these tools

In [23]:
response = model.invoke([HumanMessage(content="TELL ME THE CLOSING PRICE OF RELIANCE STOCK YESTERDAY WITH THE DATE")])
response.content

"I'm not able to provide real-time or the most recent stock prices. However, I can suggest some ways for you to find the closing price of Reliance stock yesterday.\n\nYou can check the official website of the National Stock Exchange (NSE) or the Bombay Stock Exchange (BSE) in India for the latest stock prices. Alternatively, you can also check financial news websites or mobile apps such as MoneyControl, Bloomberg, or Reuters for the latest stock prices.\n\nIf you're looking for historical stock price data, I can suggest some sources such as Yahoo Finance or Quandl, which provide historical stock price data for various companies, including Reliance Industries Limited.\n\nPlease note that the stock market is subject to fluctuations, and the closing price of Reliance stock may change from day to day. If you're looking for the most recent closing price, I recommend checking a reliable financial news source or the official website of the stock exchange.\n\nAs I'm a large language model, my 

### CONNECT TO A TOOL

In [26]:
#bind_tools to give model knowledge about these tools

model_with_tools = model.bind_tools(tools)

We can now call the model. Let's first call it with a normal message, and see how it responds. We can look at both the content field as well as the tool_calls field.

In [28]:
#response on some normal message
response = model_with_tools.invoke([HumanMessage(content="Hi!")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: It's nice to meet you. Is there something I can help you with or would you like to chat?
ToolCalls: []


Now, let's try calling it with some input that would expect a tool to be called.

In [30]:
#trying on some thing that will call the tool

response = model_with_tools.invoke([HumanMessage(content="Yesterday I met my freind we had a discussion on cricket")])
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: It sounds like you had a fun conversation with your friend about cricket! Did you discuss any particular teams, players, or recent matches?
ToolCalls: []


In [32]:
#trying on some thing that will call the tool

response = model_with_tools.invoke([HumanMessage(content='what is the closing price of reliance stock yesterday?')])
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': 'reliance stock closing price yesterday'}, 'id': 'call_0ep5pa32ctj4ss9eyopk2gk0', 'type': 'tool_call'}]


We can see that there's now **no text content**, but there is a **tool call**! ``It wants us to call the Tavily Search tool.``

``This isn't calling that tool yet`` - it's just telling us to. In order to actually call it, we'll want to create our agent

# Create the agent

* Now that we have defined the tools and the LLM, we can create the agent. We will be using LangGraph to construct the agent. Currently we are using a high level interface to construct the agent, but the nice thing about LangGraph is that this high-level interface is backed by a low-level, highly controllable API in case you want to modify the agent logic.

* Now, we can initialize the agent with the LLM and the tools.

* Note that we are passing in the model, not model_with_tools. That is because create_react_agent will call .bind_tools for us under the hood.



In [34]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

# Run the agent

Overall, this line of code sets up an agent that can leverage a language model and additional tools to perform tasks dynamically, such as responding to user queries or performing specific actions based on the inputs it receives

In [39]:
response = agent_executor.invoke({"messages": HumanMessage(content="hi!")})

response["messages"][1].content


"It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [41]:
response = agent_executor.invoke({"messages": HumanMessage(content="how is going today!")})

response["messages"][1].content

"I'm doing well, thanks for asking! Is there something I can help you with or would you like to chat?"

In [43]:
response = agent_executor.invoke({"messages": HumanMessage(content="Yesterday I met my freind and we had discussion about cricket")})

response["messages"]

[HumanMessage(content='Yesterday I met my freind and we had discussion about cricket', additional_kwargs={}, response_metadata={}, id='b7eb6349-3cd2-4a2a-98c1-5c75fc82ed30'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_dkdlw0pvxpytjunq2mm5s9hk', 'function': {'arguments': '{"query":"latest news in cricket"}', 'name': 'tavily_search_results_json'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 351, 'total_tokens': 372, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-b5756dbb-eb59-4421-ac97-098600d3285a-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'latest news in cricket'}, 'id': 'call_dkdlw0pvxpytjunq2mm5s9hk', 'type': 'tool_call'}], usage_metadata={'input_tokens': 351, 'output_tokens': 21

In [45]:
response = agent_executor.invoke(
    {"messages": HumanMessage(content="what is the closing price of reliance stock yesterday?")}
)
response["messages"]

[HumanMessage(content='what is the closing price of reliance stock yesterday?', additional_kwargs={}, response_metadata={}, id='9a89e2bd-d886-46a0-b62f-cfc4b5becb00'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_wxrtgxc0nw1zpj7xbml083jm', 'function': {'arguments': '{"query":"reliance stock closing price yesterday"}', 'name': 'tavily_search_results_json'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 349, 'total_tokens': 373, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-65689e0e-62e1-458e-bffd-4a5289c521d4-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'reliance stock closing price yesterday'}, 'id': 'call_wxrtgxc0nw1zpj7xbml083jm', 'type': 'tool_call'}], usage_metadata={'input_tokens':

In [61]:
type(response['messages'])

list

In [59]:
response['messages'][3].content

'The closing price of Reliance stock yesterday was ₹1,281.55.'

In [121]:
response = agent_executor.invoke(
    {"messages": [HumanMessage(content="today was a great day")]}
)
response["messages"]

[HumanMessage(content='today was a great day', additional_kwargs={}, response_metadata={}, id='4e4c3b90-df7f-4dbd-aa6f-230c3c0f6705'),
 AIMessage(content=' I\'m sorry, but I currently don\'t have the capability to evaluate or provide opinions on statements like "today was a great day". My main function is to perform tasks such as running a search query. If you need any information on a specific topic, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 192, 'total_tokens': 252, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'system_fingerprint': None, 'finish_reason': 'eos', 'logprobs': None}, id='run-07c4365a-aec6-4200-a2b8-f79b208510a0-0', usage_metadata={'input_tokens': 192, 'output_tokens': 60, 'total_tokens': 252, 'input_token_details': {}, 'output_token_details': {}})]

In [123]:
response = agent_executor.invoke(
    {"messages": [HumanMessage(content="Tell me the todays stock closing proce of Reliance, ICICI and Adani power")]}
)
response["messages"]

[HumanMessage(content='Tell me the todays stock closing proce of Reliance, ICICI and Adani power', additional_kwargs={}, response_metadata={}, id='61838a39-eb0b-4114-9afe-7ff1b4d631c6'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_lwvo9rg0oe7278dmz8zph2xh', 'function': {'arguments': '{"query":"todays stock closing price of Reliance, ICICI and Adani power"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 206, 'total_tokens': 265, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-ef78e922-db8e-476e-a5c8-d70d8358ff11-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'todays stock closing price of Reliance, ICICI and Adani power'}, 'id': 'call_lwvo9rg0oe7278dmz8zph2xh', 'type': '

In [124]:
type(response["messages"][2].content)

str

In [125]:
response["messages"][2].content

'[{"url": "https://in.investing.com/equities/reliance-industries/1000", "content": "What Is the Reliance Industries Ltd Stock Price Today? The Reliance Industries Ltd stock price today is 1,291.70. ... India equities were higher at the close on Tuesday, as gains in the Auto, Real Estate and Consumer Durables sectors led shares higher. ... Adani Power. 561.65 +0.12%: 34.52M. Adani Power. 561.80 +0.14%: 34.49M. Adani Energy"}, {"url": "https://www.livemint.com/market/market-stats/stocks-adani-power-share-price-nse-bse-s0003083", "content": "Adani Power. is trading-0.57% lower at Rs 536.95 as compared to its last closing price.. Adani Power has been trading in the price range of 544.95 & 531.90. Adani Power has given 2.80% in this"}, {"url": "https://www.business-standard.com/markets/reliance-industries-ltd-share-price-476.html", "content": "RIL Share Price Today - Reliance Industries Stock Price Live NSE/BSE | Business Standard Reliance Industries Ltd 1. What\'s the Reliance Industries L

# Streaming

In [130]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats the weather in Delhi?")]}
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_b4ps204pbczsb0jw214xgkx0', 'function': {'arguments': '{"query":"weather in Delhi"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 193, 'total_tokens': 239, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-0dd332d3-f468-4b66-b2aa-a3279fd34707-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Delhi'}, 'id': 'call_b4ps204pbczsb0jw214xgkx0', 'type': 'tool_call'}], usage_metadata={'input_tokens': 193, 'output_tokens': 46, 'total_tokens': 239, 'input_token_details': {}, 'output_token_details': {}})]}}
----
{'tools': {'messages': [ToolMessage(content='[{"url": "https://www.weatherapi.com/", "content": "

# Adding Memory

As mentioned earlier, this agent is stateless. This means it does not remember previous interactions. To give it memory we need to pass in a checkpointer. When passing in a checkpointer, we also have to pass in a thread_id when invoking the agent (so it knows which thread/conversation to resume from).

In [79]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [81]:
agent_executor = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "123"}}

In [83]:
for chunk in agent_executor.stream({"messages": [HumanMessage(content="hi my name is bob!")]}, config):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 345, 'total_tokens': 371, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-79f7d5e1-6cdf-4c33-a345-fa0cee72cc63-0', usage_metadata={'input_tokens': 345, 'output_tokens': 26, 'total_tokens': 371, 'input_token_details': {}, 'output_token_details': {}})]}}
----


In [84]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats my name?")]}, config):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='Your name is Bob.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 385, 'total_tokens': 391, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-405aabf7-c787-4e3c-9126-49e9fe236232-0', usage_metadata={'input_tokens': 385, 'output_tokens': 6, 'total_tokens': 391, 'input_token_details': {}, 'output_token_details': {}})]}}
----


In [85]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="WHAT IS THE RESULT OF NEW XEALAND VS PAKISTAN MATCH")]}, config):
    print(chunk)
    print("------------------------------")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_fhhdy3h8bfw2loi8t90jjzqq', 'function': {'arguments': '{"query":"New Zealand vs Pakistan match result"}', 'name': 'tavily_search_results_json'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 416, 'total_tokens': 439, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-ed0e3ab4-d552-4db8-be78-9673c54b7b4d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'New Zealand vs Pakistan match result'}, 'id': 'call_fhhdy3h8bfw2loi8t90jjzqq', 'type': 'tool_call'}], usage_metadata={'input_tokens': 416, 'output_tokens': 23, 'total_tokens': 439, 'input_token_details': {}, 'output_token_details': {}})]}}
------------------------------
{'tools': {

In [88]:
m=[]
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="WHAT IS THE RESULT OF NEW XEALAND VS PAKISTAN MATCH")]}, config):
    m.append(chunk)
    
    print("----")
m[2].content

----
----
----


AttributeError: 'AddableUpdatesDict' object has no attribute 'content'

* If I want to start a new conversation, all I have to do is change the thread_id used

In [147]:
#different thread_id
config = {"configurable": {"thread_id": "xyz123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="Janet is so cool")]}, config
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content=" That's great! How can I assist you further with your search?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 191, 'total_tokens': 207, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'system_fingerprint': None, 'finish_reason': 'eos', 'logprobs': None}, id='run-c3a6d230-3b28-4801-930d-598d62b20820-0', usage_metadata={'input_tokens': 191, 'output_tokens': 16, 'total_tokens': 207, 'input_token_details': {}, 'output_token_details': {}})]}}
----


In [149]:
config = {'configurable':{'thread_id':"abc123"}}

for chunk in agent_executor.stream({"messages":[HumanMessage(content='Bob wants to know the last test match result of ind vs bangladesh')]},
                                  config):
    print(chunk)
    
   

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_znaxyqtg6syldefqjfuaa201', 'function': {'arguments': '{"query":"last test match result of ind vs bangladesh"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 203, 'total_tokens': 256, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-cd1c3373-9303-4c24-baf4-d6bd0918e023-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'last test match result of ind vs bangladesh'}, 'id': 'call_znaxyqtg6syldefqjfuaa201', 'type': 'tool_call'}], usage_metadata={'input_tokens': 203, 'output_tokens': 53, 'total_tokens': 256, 'input_token_details': {}, 'output_token_details': {}})]}}
{'tools': {'messages': [ToolMessage(content='[{"

In [72]:
for i in agent_executor.stream({"messages":[HumanMessage(content='what is the latest news on Sensex')]},
                                  config):
    print(i)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_IZHZoPQJ25OgKAmZLPNLOmoP', 'function': {'arguments': '{"query": "latest news on Sensex"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 1226, 'total_tokens': 1246, 'completion_tokens_details': {'reasoning_tokens': 0}}, 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-224ec767-1688-49af-ac1a-9cca64cfa3bd-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'latest news on Sensex'}, 'id': 'call_IZHZoPQJ25OgKAmZLPNLOmoP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1226, 'output_tokens': 20, 'total_tokens': 1246})]}}
{'tools': {'messages': [ToolMessage(content='[{"url": "https://economictimes.indiatimes.com/markets/stocks/live-blog/bse-sensex-today-live-nifty-stock-market-updates-25-september-2